In [ ]:
# Data Wrangling
import pandas as pd
from pandas import Series, DataFrame
import numpy as np

# Visualization
import matplotlib.pylab as plt
from matplotlib import font_manager, rc
import seaborn as sns
%matplotlib inline
plt.rc('font',family='Malgun Gothic')
from matplotlib import font_manager, rc
# EDA
import klib

# Data Split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
seed = 42
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)

# Preprocessing & Feature Engineering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import SelectPercentile
from sklearn import model_selection

# Hyperparameter Optimization
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from bayes_opt import BayesianOptimization

# Modeling
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.base import ClassifierMixin

# Evaluation
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Utility
import os
import time
import random
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image
import pickle
from tqdm import tqdm, tqdm_notebook
import platform
from itertools import combinations
from scipy.stats.mstats import gmean
import warnings

In [ ]:
train = pd.read_csv('data/data_01_01.csv')
test = pd.read_csv('data/data_02_02.csv')
label = pd.read_csv('data/data_03_03.csv')

In [ ]:
q = test.cust.unique()
train['y_train'] = train.cust.apply(lambda x:1 if x in q else 0)

In [ ]:
q = label.cust.unique()
test['y_test'] = test.cust.apply(lambda x:1 if x in q else 0)

In [ ]:
set(train.columns)- set(test.columns)

In [ ]:
data_c = pd.concat([train, test])

In [ ]:
주구매 = data_c.filter(regex = '주구매상품대분류').columns.to_list()

data_c.drop(주구매,axis = 1,inplace  = True)

In [ ]:
train = data_c.iloc[:len(train)]
test = data_c.iloc[len(train):]

train.most_cop = train.most_cop.fillna('X')
test.most_cop = test.most_cop.fillna('X')

In [ ]:
train.most_chnl = train.most_chnl.fillna(0)
test.most_chnl = test.most_chnl.fillna(0)

train.point_count = train.point_count.fillna(train.point_count.mean())
test.point_count = test.point_count.fillna(test.point_count.mean())

train.point_sum = train.point_count.fillna(train.point_sum.mean())
test.point_sum = test.point_count.fillna(test.point_sum.mean())

In [ ]:
train = train.fillna(0)
test = test.fillna(0)

In [ ]:
y_train = train.y_train
del(train['y_train'])
y_test = test.y_test
del(test['y_test'])

In [ ]:
train_id=train.cust
del(train['cust'])
test_id=test.cust
del(test['cust'])

In [ ]:
del train["y_test"]
del test["y_train"]

In [ ]:
train=pd.get_dummies(data = train, columns = ['most_cop'], prefix = 'most_cop')
test=pd.get_dummies(data = test, columns = ['most_cop'], prefix = 'most_cop')

In [ ]:
X_train = train
X_test = test

In [ ]:
# standard Scaler 정규화 진행

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# 군집화 진행   # 5로 했을 때 성능이 좋아짐.  # 5, 10만 실험해봄,

from sklearn.cluster import KMeans
model = KMeans(n_clusters=5)
model.fit(X_train)

In [ ]:
X_train=pd.DataFrame(X_train)
X_test=pd.DataFrame(X_test)

In [ ]:
# 군집화 결과 피쳐 추가

X_train["group"]=model.predict(X_train)
X_test["group"]=model.predict(X_test)

In [ ]:
X_train.group.value_counts()

In [ ]:
train['group'] = X_train["group"]
test['group'] = X_test["group"]

In [ ]:
three = pd.read_csv('submissions/03.csv')

In [ ]:
test = pd.concat([test,three], axis = 1)

In [ ]:
test

In [ ]:
group_0 = test.query("STATUS < 0.5")
group_1 = test.query("STATUS >= 0.5 & STATUS < 0.7")
group_2 = test.query("STATUS >= 0.7 & STATUS < 0.9")
group_3 = test.query("STATUS >= 0.9")

In [ ]:
test.groupby("group")["STATUS"].mean()

In [ ]:
ct_group_0 = test.query("group == 3")
ct_group_1 = test.query("group == 2")
ct_group_2 = test.query("group == 4")
ct_group_3 = test.query("group == 1")
ct_group_4 = test.query("group == 0")

## 시각화

In [ ]:
group_name=["취약 고객","중립 고객1","중립 고객2","충성 고객"]
group_lst=[group_0,group_1,group_2,group_3]
for c in test.columns.to_list()[:-3]:
    dt=[]
    for g in group_lst:
        dt.append(g[c].mean())
    plt.bar(group_name,dt,color=["gray", "orange", "orange", "royalblue"], width=0.5)
    plt.title(c)
    plt.show()

In [ ]:
ct_group_name=["ct_group_0","ct_group_1","ct_group_2","ct_group_3", "ct_group_4"]
ct_group_lst=[ct_group_0,ct_group_1,ct_group_2,ct_group_3, ct_group_4]
for c in test.columns.to_list()[:-3]:
    dt=[]
    for g in ct_group_lst:
        dt.append(g[c].mean())
    plt.bar(ct_group_name,dt, width=0.5)
    plt.title(c)
    plt.show()

In [ ]:
group_0 = ct_group_0.query("STATUS < 0.5")
group_1 = ct_group_0.query("STATUS >= 0.5 & STATUS < 0.7")
group_2 = ct_group_0.query("STATUS >= 0.7 & STATUS < 0.9")
group_3 = ct_group_0.query("STATUS >= 0.9")

print("다음 달에 올 확률에 따른 고객 군집 별 수")
print("취약 고객 : ",len(group_0),"중립 고객1 : ", len(group_1),"중립 고객2 : ", len(group_2),"충성 고객 : ", len(group_3))

group_name=["취약 고객","중립 고객1","중립 고객2","충성 고객"]
group_lst=[group_0,group_1,group_2,group_3]
for c in test.columns.to_list()[:-3]:
    dt=[]
    for g in group_lst:
        dt.append(g[c].mean())
    plt.bar(group_name,dt,color=["gray", "orange", "orange", "royalblue"], width=0.5)
    plt.title("CLUSTER 3  "+c)
    plt.show()

In [ ]:
group_0 = ct_group_1.query("STATUS < 0.5")
group_1 = ct_group_1.query("STATUS >= 0.5 & STATUS < 0.7")
group_2 = ct_group_1.query("STATUS >= 0.7 & STATUS < 0.9")
group_3 = ct_group_1.query("STATUS >= 0.9")

print("다음 달에 올 확률에 따른 고객 군집 별 수")
print("취약 고객 : ",len(group_0),"중립 고객1 : ", len(group_1),"중립 고객2 : ", len(group_2),"충성 고객 : ", len(group_3))

group_name=["취약 고객","중립 고객1","중립 고객2","충성 고객"]
group_lst=[group_0,group_1,group_2,group_3]
for c in test.columns.to_list()[:-3]:
    dt=[]
    for g in group_lst:
        dt.append(g[c].mean())
    plt.bar(group_name,dt,color=["gray", "orange", "orange", "royalblue"], width=0.5)
    plt.title("CLUSTER 2  "+c)
    plt.show()

In [ ]:
group_0 = ct_group_2.query("STATUS < 0.5")
group_1 = ct_group_2.query("STATUS >= 0.5 & STATUS < 0.7")
group_2 = ct_group_2.query("STATUS >= 0.7 & STATUS < 0.9")
group_3 = ct_group_2.query("STATUS >= 0.9")

print("다음 달에 올 확률에 따른 고객 군집 별 수")
print("취약 고객 : ",len(group_0),"중립 고객1 : ", len(group_1),"중립 고객2 : ", len(group_2),"충성 고객 : ", len(group_3))

group_name=["취약 고객","중립 고객1","중립 고객2","충성 고객"]
group_lst=[group_0,group_1,group_2,group_3]
for c in test.columns.to_list()[:-3]:
    dt=[]
    for g in group_lst:
        dt.append(g[c].mean())
    plt.bar(group_name,dt,color=["gray", "orange", "orange", "royalblue"], width=0.5)
    plt.title("CLUSTER 4  "+c)
    plt.show()

In [ ]:
group_0 = ct_group_3.query("STATUS < 0.5")
group_1 = ct_group_3.query("STATUS >= 0.5 & STATUS < 0.7")
group_2 = ct_group_3.query("STATUS >= 0.7 & STATUS < 0.9")
group_3 = ct_group_3.query("STATUS >= 0.9")

print("다음 달에 올 확률에 따른 고객 군집 별 수")
print("취약 고객 : ",len(group_0),"중립 고객1 : ", len(group_1),"중립 고객2 : ", len(group_2),"충성 고객 : ", len(group_3))

group_name=["취약 고객","중립 고객1","중립 고객2","충성 고객"]
group_lst=[group_0,group_1,group_2,group_3]
for c in test.columns.to_list()[:-3]:
    dt=[]
    for g in group_lst:
        dt.append(g[c].mean())
    plt.bar(group_name,dt,color=["gray", "orange", "orange", "royalblue"], width=0.5)
    plt.title("CLUSTER 1  "+c)
    plt.show()

In [ ]:
group_0 = ct_group_3.query("STATUS < 0.5")
group_1 = ct_group_3.query("STATUS >= 0.5 & STATUS < 0.7")
group_2 = ct_group_3.query("STATUS >= 0.7 & STATUS < 0.9")
group_3 = ct_group_3.query("STATUS >= 0.9")

print("다음 달에 올 확률에 따른 고객 군집 별 수")
print("취약 고객 : ",len(group_0),"중립 고객1 : ", len(group_1),"중립 고객2 : ", len(group_2),"충성 고객 : ", len(group_3))

group_name=["취약 고객","중립 고객1","중립 고객2","충성 고객"]
group_lst=[group_0,group_1,group_2,group_3]
for c in test.columns.to_list()[:-3]:
    dt=[]
    for g in group_lst:
        dt.append(g[c].mean())
    plt.bar(group_name,dt,color=["gray", "orange", "orange", "royalblue"], width=0.5)
    plt.title("CLUSTER 0  "+c)
    plt.show()